<a href="https://colab.research.google.com/github/cncPomper/MMC/blob/master/lab_5/MMC_lab_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from forest_fire_percolation_modified import *

In [ ]:
# Parametry symulacji:
par = ParametryPozaru(p=0.01, f=1e-5, q=1.0)

# Dla celów laboratoryjnych dobieramy L i liczbę kroków pod czas obliczeń.
L = 120

# Dla animacji wygodnie jest zacząć od niezerowej gęstości (żeby pożary pojawiły się szybciej).
gestosc_start = 0.55

ile_krokow = 800
kroki_na_klatke = 2  # odpowiednik "steps per frame"
seed = 2025

las = Las(L, seed=seed)
las.inicjalizuj_losowo(gestosc_start)

stat = nagraj_animacje(
  las,
  par,
  ile_krokow=ile_krokow,
  plik_wyj="pozar.gif",
  fps=20,
  co_ile_krokow=kroki_na_klatke,
)
stat.zapisz_do_pliku("statystyki_pozaru.txt")
las.zapisz_do_pliku("mapa_koncowa.txt")

wykres_gestosci_w_czasie(stat, "gestosc_w_czasie.png")
wykres_rozmiarow_pozarow(stat, "rozmiary_pozarow_loglog.png")

krzywa_perkolacji(L=128, proby_na_punkt=60, plik="krzywa_perkolacji.png")

print("Zapisano pliki: pozar.gif, statystyki_pozaru.txt, mapa_koncowa.txt, gestosc_w_czasie.png, "
"rozmiary_pozarow_loglog.png, krzywa_perkolacji.png")

Zapisano pliki: pozar.gif, statystyki_pozaru.txt, mapa_koncowa.txt, gestosc_w_czasie.png, rozmiary_pozarow_loglog.png, krzywa_perkolacji.png


In [ ]:
# Te same parametry bazowe, ale teraz będziemy zmieniać q, żeby zobaczyć jego wpływ na dynamikę pożaru i perkolację.
# L = 120
# gestosc_start = 0.55
# ile_krokow = 800
# kroki_na_klatke = 2
# seed = 2025

# Wartości q do przetestowania zgodnie z poleceniem
wartosci_q = [1.0, 0.6, 0.3]
wszystkie_statystyki: Dict[float, Statystyka] = {}

for q in wartosci_q:
    print(f"Uruchamianie symulacji dla q = {q}...")

    # Konfiguracja parametrów dla bieżącego przebiegu
    par = ParametryPozaru(p=0.01, f=1e-5, q=q)

    # Inicjalizacja lasu zawsze z tym samym seedem, aby warunki początkowe były identyczne
    las = Las(L, seed=seed)
    las.inicjalizuj_losowo(gestosc_start)

    # Rejestracja animacji dla danego q
    nazwa_animacji = f"pozar_q_{q:.1f}.gif"
    stat = nagraj_animacje(
        las,
        par,
        ile_krokow=ile_krokow,
        plik_wyj=nazwa_animacji,
        fps=20,
        co_ile_krokow=kroki_na_klatke,
        tytul=f"Pożar lasu (model DS) | q = {q}",
    )

    # Zapis statystyk dla tego konkretnego q
    stat.zapisz_do_pliku(f"statystyki_q_{q:.1f}.txt")
    wszystkie_statystyki[q] = stat

# Generowanie wykresu porównawczego dla wszystkich przebiegów q
wykres_porownawczy_q(wszystkie_statystyki, "porownanie_q_w_czasie.png")

# Dodatkowo odpalamy klasyczną statyczną perkolację (niezależną od q)
print("Generowanie klasycznej krzywej perkolacji...")
krzywa_perkolacji(L=128, proby_na_punkt=60, plik="krzywa_perkolacji.png")

print("\n[Zakończono] Wygenerowano pliki:")
for q in wartosci_q:
    print(f" - pozar_q_{q:.1f}.gif oraz statystyki_q_{q:.1f}.txt")
print(" - porownanie_q_w_czasie.png (kluczowy wykres do analizy progu)")
print(" - krzywa_perkolacji.png")

Uruchamianie symulacji dla q = 1.0...
Uruchamianie symulacji dla q = 0.6...
Uruchamianie symulacji dla q = 0.3...
Generowanie klasycznej krzywej perkolacji...

[Zakończono] Wygenerowano pliki:
 - pozar_q_1.0.gif oraz statystyki_q_1.0.txt
 - pozar_q_0.6.gif oraz statystyki_q_0.6.txt
 - pozar_q_0.3.gif oraz statystyki_q_0.3.txt
 - porownanie_q_w_czasie.png (kluczowy wykres do analizy progu)
 - krzywa_perkolacji.png


##Na podstawie zebranych danych ze statystyk widać bardzo wyraźnie, w jaki sposób zmiana parametru q (najprawdopodobniej określającego prawdopodobieństwo przeniesienia się ognia na sąsiednie drzewo) wpływa na dynamikę lasu i pożaru:

### Dla q = 1.0:

- Pożar wykazuje bardzo agresywny i gwałtowny rozwój.

### Dla q = 0.6:

- Pożar wciąż obejmuje znaczne części lasu, ale proces ten jest zauważalnie spowolniony i wypala las w mniejszym stopniu Oznacza to łagodniejszy i wolniejszy przebieg destrukcyjny.

### Dla q = 0.3:

- System przechodzi całkowitą zmianę charakterystyki, prawdopodobnie wchodząc w stan poniżej tzw. progu perkolacji. Ogień z bardzo dużym trudem przeskakuje na sąsiadów, co sprawia, że większość ognisk błyskawicznie i samoistnie wygasa. Przy tym ustawieniu las jest mocno chroniony przed wyginięciem w wielkoobszarowym pożarze.

## Wniosek ogólny:
- Zmniejszanie parametru q spowalnia dynamikę rozszerzania się pożaru, zmniejszając jego całkowite żniwo. Zmniejszenie pożaru do poziomu 0.3 drastycznie hamuje jego zasięg, udaremniając powstanie gigantycznego, samonapędzającego się ogniska.